In [1]:
import os
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
print(PROJECT_ROOT)
print(list(DATA_RAW.iterdir()))

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("otto-explore")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
print(spark.version)

/Users/os_bhosale/Desktop/Data Science Projects/e-commerce recommender
[PosixPath('/Users/os_bhosale/Desktop/Data Science Projects/e-commerce recommender/data/raw/otto-recsys-train.jsonl'), PosixPath('/Users/os_bhosale/Desktop/Data Science Projects/e-commerce recommender/data/raw/otto-recsys-test.jsonl')]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 11:00:25 WARN Utils: Your hostname, Omkars-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.75 instead (on interface en0)
26/11/25 11:00:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 11:00:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.3.0


In [2]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("session", T.LongType()),
    T.StructField("events", T.ArrayType(T.StructType([
        T.StructField("aid", T.LongType()),
        T.StructField("ts", T.LongType()),
        T.StructField("type", T.StringType()),
    ]))),
])

sessions = spark.read.schema(schema).json(str(DATA_RAW / "otto-recsys-train.jsonl"))
sessions.printSchema()
sessions.show(3, truncate=80)

root
 |-- session: long (nullable = true)
 |-- events: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- aid: long (nullable = true)
 |    |    |-- ts: long (nullable = true)
 |    |    |-- type: string (nullable = true)

+-------+--------------------------------------------------------------------------------+
|session|                                                                          events|
+-------+--------------------------------------------------------------------------------+
|      0|[{1517085, 1659304800025, clicks}, {1563459, 1659304904511, clicks}, {1309446...|
|      1|[{424964, 1659304800025, carts}, {1492293, 1659304852871, clicks}, {1492293, ...|
|      2|[{763743, 1659304800038, clicks}, {137492, 1659304866606, clicks}, {504789, 1...|
+-------+--------------------------------------------------------------------------------+
only showing top 3 rows


In [3]:
from pyspark.sql import functions as F

events = (
    sessions
    .select("session", F.explode("events").alias("e"))
    .select("session", "e.aid", "e.ts", "e.type")
)

events.groupBy("type").count().orderBy(F.desc("count")).show()

+------+---------+
|  type|    count|
+------+---------+
|clicks|194720954|
| carts| 16896191|
|orders|  5098951|
+------+---------+



In [4]:
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

type_map = F.create_map(
    F.lit("clicks"), F.lit(0),
    F.lit("carts"), F.lit(1),
    F.lit("orders"), F.lit(2),
)

events_compact = events.select(
    F.col("session").cast("int"),
    F.col("aid").cast("int"),
    (F.col("ts") / 1000).cast("int").alias("ts"),
    type_map[F.col("type")].cast("byte").alias("type"),
)

events_compact.write.mode("overwrite").parquet(str(DATA_PROCESSED / "train_events"))
print("saved")

saved


In [5]:
ev = spark.read.parquet(str(DATA_PROCESSED / "train_events"))
print(ev.count())
ev.show(5)

216716096
+-------+-------+----------+----+
|session|    aid|        ts|type|
+-------+-------+----------+----+
|6532251| 752652|1660149825|   0|
|6532251|1805845|1660149885|   0|
|6532251| 752652|1660149897|   0|
|6532251| 950341|1660149907|   0|
|6532251|1779103|1660150049|   0|
+-------+-------+----------+----+
only showing top 5 rows


In [7]:
daily = (
    ev.withColumn("date", F.to_date(F.from_unixtime("ts")))
    .groupBy("date")
    .agg(
        F.count("*").alias("events"),
        F.countDistinct("session").alias("sessions"),
    )
    .orderBy("date")
)
daily.show(40)

ev.select(
    F.from_unixtime(F.min("ts")).alias("first_event"),
    F.from_unixtime(F.max("ts")).alias("last_event"),
).show()

+----------+--------+--------+
|      date|  events|sessions|
+----------+--------+--------+
|2022-07-31|  554578|   98476|
|2022-08-01| 7605240| 1220697|
|2022-08-02| 7522151| 1262919|
|2022-08-03| 7961997| 1324843|
|2022-08-04| 7666071| 1285931|
|2022-08-05| 7060736| 1158915|
|2022-08-06| 6912912| 1107510|
|2022-08-07| 8351180| 1223776|
|2022-08-08| 6818399| 1080483|
|2022-08-09| 7980270| 1240788|
|2022-08-10| 7307510| 1222426|
|2022-08-11| 7317690| 1229569|
|2022-08-12| 7574703| 1287162|
|2022-08-13| 7984390| 1325347|
|2022-08-14|10493076| 1630069|
|2022-08-15| 8831665| 1475326|
|2022-08-16| 7675104| 1271411|
|2022-08-17| 7661820| 1266228|
|2022-08-18| 7024989| 1210177|
|2022-08-19| 7330377| 1227835|
|2022-08-20| 7752935| 1245160|
|2022-08-21| 9049680| 1383552|
|2022-08-22| 7262589| 1221438|
|2022-08-23| 7253694| 1232761|
|2022-08-24| 6934775| 1194705|
|2022-08-25| 6513549| 1143867|
|2022-08-26| 7293070| 1160633|
|2022-08-27| 7825435| 1186802|
|2022-08-28| 9195511| 1328009|
+-------

+-------------------+-------------------+
|        first_event|         last_event|
+-------------------+-------------------+
|2022-07-31 18:00:00|2022-08-28 17:59:59|
+-------------------+-------------------+



In [8]:
max_ts = ev.agg(F.max("ts")).first()[0]
cutoff = max_ts - 7 * 24 * 3600 + 1
print("cutoff:", cutoff)

train = ev.filter(F.col("ts") < cutoff)

session_start = ev.groupBy("session").agg(F.min("ts").alias("start_ts"))
test_sessions = session_start.filter(F.col("start_ts") >= cutoff).select("session")
test = ev.join(test_sessions, "session")

train.write.mode("overwrite").parquet(str(DATA_PROCESSED / "split_train"))
test.write.mode("overwrite").parquet(str(DATA_PROCESSED / "split_test"))

tr = spark.read.parquet(str(DATA_PROCESSED / "split_train"))
te = spark.read.parquet(str(DATA_PROCESSED / "split_test"))

for name, df in [("train", tr), ("test", te)]:
    print(name, "events:", df.count(), "sessions:", df.select("session").distinct().count())
    df.groupBy("type").count().orderBy("type").show()

cutoff: 1661119200


train events: 163955181 sessions: 11098528


+----+---------+
|type|    count|
+----+---------+
|   0|147407286|
|   1| 12681933|
|   2|  3865962|
+----+---------+

test events: 15384577 sessions: 1801251
+----+--------+
|type|   count|
+----+--------+
|   0|13664631|
|   1| 1312541|
|   2|  407405|
+----+--------+



In [9]:
from pyspark.sql import Window

w = Window.partitionBy("session").orderBy("ts", "aid")
te_n = te.withColumn("idx", F.row_number().over(w) - 1)

cuts = (
    te_n.groupBy("session").agg(F.count("*").alias("n"))
    .filter(F.col("n") >= 2)
    .withColumn("cut", (F.floor(F.rand(seed=42) * (F.col("n") - 1)) + 1).cast("int"))
)

joined = te_n.join(cuts, "session")
test_input = joined.filter(F.col("idx") < F.col("cut")).select("session", "aid", "ts", "type")
after = joined.filter(F.col("idx") >= F.col("cut"))

next_click = (
    after.filter(F.col("type") == 0)
    .groupBy("session").agg(F.min_by("aid", "idx").alias("click"))
)
carts = (
    after.filter(F.col("type") == 1)
    .groupBy("session").agg(F.collect_set("aid").alias("carts"))
)
orders = (
    after.filter(F.col("type") == 2)
    .groupBy("session").agg(F.collect_set("aid").alias("orders"))
)

labels = (
    cuts.select("session")
    .join(next_click, "session", "left")
    .join(carts, "session", "left")
    .join(orders, "session", "left")
)

test_input.write.mode("overwrite").parquet(str(DATA_PROCESSED / "test_input"))
labels.write.mode("overwrite").parquet(str(DATA_PROCESSED / "test_labels"))

ti = spark.read.parquet(str(DATA_PROCESSED / "test_input"))
tl = spark.read.parquet(str(DATA_PROCESSED / "test_labels"))

print("input events:", ti.count(), "sessions:", ti.select("session").distinct().count())
print("label sessions:", tl.count())
print("with a next click:", tl.filter(F.col("click").isNotNull()).count())
print("with carts:", tl.filter(F.col("carts").isNotNull()).count())
print("with orders:", tl.filter(F.col("orders").isNotNull()).count())
tl.show(5, truncate=60)

input events: 7691780 sessions: 1801251
label sessions: 1801251
with a next click: 1755489
with carts: 306859
with orders: 151078
+--------+-------+---------+---------+
| session|  click|    carts|   orders|
+--------+-------+---------+---------+
|11098532| 461190|     NULL|     NULL|
|11098540|  87003|     NULL|     NULL|
|11098544|1596897|     NULL|     NULL|
|11098546| 152419|[1541643]|[1541643]|
|11098549|1672025|     NULL|     NULL|
+--------+-------+---------+---------+
only showing top 5 rows
